In [21]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np
import calendar
import time
from datetime import datetime
from dateutil.relativedelta import relativedelta
import warnings
warnings.filterwarnings('ignore')

# 프로젝트 경로 설정
here = Path.cwd()
project_path = None
for p in [here, *here.parents]:
    if (p / "DATA").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        project_path = str(p)
        break

if project_path is None:
    fallback = r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast"
    if os.path.isdir(fallback) and fallback not in sys.path:
        sys.path.insert(0, fallback)
    project_path = fallback

print("Using project path:", project_path)

from DATA.stock_invest_function import *
import importlib
import DATA.us_sarima_forecast as sarima
importlib.reload(sarima)
import DATA.us_lstm_forecast_v2 as lstm_v2
importlib.reload(lstm_v2)
import DATA.us_prophet_forecast_v3 as prophet_v3
importlib.reload(prophet_v3)
import DATA.us_est_forecast_v2 as esmod
importlib.reload(esmod)

# ========================
# 유틸리티 함수
# ========================


# 유틸리티 함수들
def convert_to_month_end(date_str):
    try:
        # 문자열/타입 혼용 안전 변환
        date_obj = pd.to_datetime(date_str)
        if pd.isna(date_obj):
            return None

        y, m, d = date_obj.year, date_obj.month, date_obj.day

        # 1~5일 → 전달 말일
        if 1 <= d <= 5:
            if m == 1:
                prev_y, prev_m = y - 1, 12
            else:
                prev_y, prev_m = y, m - 1
            last_day_prev = calendar.monthrange(prev_y, prev_m)[1]
            return datetime(prev_y, prev_m, last_day_prev)

        # 그 외 → 해당월 말일
        last_day_cur = calendar.monthrange(y, m)[1]
        return datetime(y, m, last_day_cur)

    except Exception:
        return None


def process_daily_to_monthly_market_data(daily_data, ticker):
    if not daily_data:
        return pd.DataFrame()
    df = pd.DataFrame(daily_data)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')
    df['year_month'] = df['date'].dt.to_period('M')
    monthly_data = []
    for year_month in df['year_month'].unique():
        month_data = df[df['year_month'] == year_month]
        last_day_data = month_data.loc[month_data['date'].idxmax()]
        monthly_data.append({
            'ticker': ticker,
            'date': last_day_data['date'],
            'market_cap': last_day_data['marketCap'],
            'market_cap_billions': round(last_day_data['marketCap'] / 1_000_000_000, 2),
        })
    return pd.DataFrame(monthly_data)

def fetch_db_market_data(ticker, db_info, end_date='2024-12-31'):
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT date, ticker, me
        FROM US_fundm
        WHERE ticker = '{ticker}'
        AND me IS NOT NULL
        AND date <= '{end_date}'
        ORDER BY date ASC
        """
        df = pd.read_sql(query, con=engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df['market_cap_billions'] = df['me'] / 1000
            df['date_month_end'] = df['date'].apply(convert_date_to_month_end)
        #     convert_date_to_month_end(fmp_market_df['date']
        return df[['ticker', 'date', 'date_month_end', 'market_cap_billions']]
    except Exception as e:
        return pd.DataFrame()

def _safe_get_db_market_df(ticker, db_info):
    try:
        df = fetch_db_market_data(ticker, db_info)
        # None 이거나 길이 0이면 빈 DF 반환
        if df is None or len(df) == 0:
            return pd.DataFrame()
        return df.copy()
    except Exception as e:
        print(f"[WARN] DB 조회 중 예외 발생: {e}")
        return pd.DataFrame()


def convert_date_to_month_end(date_series):
        """날짜를 월말로 변환 (1-5일은 전월 말일로)"""
        result = []
        for dt in date_series:
            if pd.isna(dt):
                result.append(None)
                continue

            day = dt.day
            if 1 <= day <= 5:
                # 전월 말일로
                prev_month = dt - pd.offsets.MonthBegin(1)
                month_end = prev_month + pd.offsets.MonthEnd(0)
                result.append(month_end)
            else:
                # 현재월 말일로
                month_end = dt + pd.offsets.MonthEnd(0)
                result.append(month_end)

        return pd.Series(result, index=date_series.index)
# ========================
# 설정
# ========================
ticker_list = [ 'MU']

api_key = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

start_date_month = '2011-03-01'
end_date_month = (pd.Timestamp.today().normalize() - pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
measurement_date = pd.Timestamp.today().strftime('%Y-%m-%d')

print(f"\n{'='*80}")
print(f"Multi-Ticker Valuation 시작")
print(f"총 {len(ticker_list)}개 종목")
print(f"기간: {start_date_month} ~ {end_date_month}")
print(f"측정일: {measurement_date}")
print(f"{'='*80}\n")

all_results = []
success_count = 0
fail_count = 0

# ========================
# 각 ticker 순차 처리
# ========================
for idx, ticker in enumerate(ticker_list, 1):
    print(f"\n[{idx}/{len(ticker_list)}] 처리 중: {ticker}")
    print(f"\n{'='*80}")
    print(f"처리 시작: {ticker}")
    print(f"{'='*80}")

    try:
        # ========================
        # 1. FMP 매출 데이터 수집
        # ========================
        print(f"\n[{ticker}] 1. FMP 매출 데이터 수집 중...")

        url = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}"
        params = {'limit': 200, 'apikey': api_key, 'period': 'quarter'}

        response = requests.get(url, params=params, timeout=30)

        if response.status_code != 200:
            print(f"[{ticker}] ERROR: FMP 매출 데이터 수집 실패 - HTTP {response.status_code}")
            fail_count += 1
            continue

        revenue_data = response.json()

        if isinstance(revenue_data, dict) and 'Error Message' in revenue_data:
            print(f"[{ticker}] ERROR: API 오류 - {revenue_data['Error Message']}")
            fail_count += 1
            continue

        if not revenue_data:
            print(f"[{ticker}] ERROR: 데이터 없음")
            fail_count += 1
            continue

        # FMP 데이터 DataFrame 생성
        all_revenue_data = []
        for item in revenue_data:
            all_revenue_data.append({
                'ticker': ticker,
                'date': item.get('date', ''),
                'calendar_year': item.get('calendarYear', ''),
                'period': item.get('period', ''),
                'revenue': item.get('revenue', 0) if item.get('revenue') is not None else 0,
                'revenue_billions': round((item.get('revenue', 0) or 0) / 1_000_000_000, 2),
            })

        fmp_revenue_df = pd.DataFrame(all_revenue_data)
        fmp_revenue_df['date'] = pd.to_datetime(fmp_revenue_df['date'])
        fmp_revenue_df = fmp_revenue_df.sort_values(['ticker', 'date'])

        # ★ datetime 타입으로 명시적 변환
        fmp_revenue_df['date_month_end'] = pd.to_datetime(fmp_revenue_df['date'])

        print(f"\n[{ticker}] 중복 제거 전: {len(fmp_revenue_df)}건")

        # ★ 중복 제거 전 datetime 타입 보장
        fmp_revenue_df['date_month_end'] = pd.to_datetime(fmp_revenue_df['date_month_end'], errors='coerce')
        fmp_revenue_df = fmp_revenue_df.drop_duplicates(subset=['date_month_end'], keep='first').reset_index(drop=True)

        print(f"[{ticker}] 중복 제거 후 FMP 매출 데이터: {len(fmp_revenue_df)}건")

        # ========================
        # 2. DB 매출 데이터 병합
        # ========================
        print(f"\n[{ticker}] 2. DB 매출 데이터 병합 중...")

        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )

        query = f"""
        SELECT date, ticker, saleq
        FROM US_fundq
        WHERE ticker = '{ticker}'
        AND saleq IS NOT NULL
        AND date <= '2025-08-31'
        ORDER BY date ASC
        """

        db_revenue_raw = pd.read_sql(query, con=engine)
        engine.dispose()

        print(f"[{ticker}] DB 원본 데이터: {len(db_revenue_raw)}건")

        if not db_revenue_raw.empty:
            db_revenue_raw['date'] = pd.to_datetime(db_revenue_raw['date'])
            db_revenue_raw['revenue_billions'] = db_revenue_raw['saleq'] / 1000
            # ★ datetime 타입으로 명시적 변환
            db_revenue_raw['date_month_end'] = pd.to_datetime(db_revenue_raw['date'])

            # 연속 중복 제거
            db_revenue_df = db_revenue_raw.loc[
                db_revenue_raw['revenue_billions'] != db_revenue_raw['revenue_billions'].shift()
            ]
            db_revenue_df = db_revenue_df[['ticker', 'date', 'date_month_end', 'revenue_billions']]

            # ★ datetime 타입 보장
            db_revenue_df['date_month_end'] = pd.to_datetime(db_revenue_df['date_month_end'], errors='coerce')

            print(f"[{ticker}] DB 연속 중복 제거 후: {len(db_revenue_df)}건")
        else:
            db_revenue_df = pd.DataFrame()

        # ★ 병합 전 양쪽 모두 datetime 타입 보장
        fmp_revenue_df['date_month_end'] = pd.to_datetime(fmp_revenue_df['date_month_end'], errors='coerce')
        if not db_revenue_df.empty:
            db_revenue_df['date_month_end'] = pd.to_datetime(db_revenue_df['date_month_end'], errors='coerce')

        # 병합
        merged_rev_data = pd.merge(fmp_revenue_df, db_revenue_df, on=['ticker', 'date_month_end'], how='outer')
        print(f"[{ticker}] 병합 후 데이터: {len(merged_rev_data)}건")

        # 기간 필터링
        rev_data = merged_rev_data[merged_rev_data['date_month_end'] >= start_date_month].copy()
        print(f"[{ticker}] 기간 필터링 후 ({start_date_month} 이후): {len(rev_data)}건")

        # _x, _y 컬럼 병합
        if 'revenue_billions_x' in rev_data.columns and 'revenue_billions_y' in rev_data.columns:
            rev_data['revenue_billions_x'] = rev_data['revenue_billions_x'].fillna(rev_data['revenue_billions_y'])
            rev_data.rename(columns={'revenue_billions_x': 'revenue_billions'}, inplace=True)
        elif 'revenue_billions_x' in rev_data.columns:
            rev_data.rename(columns={'revenue_billions_x': 'revenue_billions'}, inplace=True)

        # rev_data['date_month_end'] = rev_data['date'].apply(convert_date_to_month_end)

        print(f"[{ticker}] 최종 데이터 건수 (정제 전): {len(rev_data)}건")

        # ========================
        # 3. 데이터 정제
        # ========================
        # revenue NaN 제거
        before = len(rev_data)
        rev_data = rev_data[~rev_data['revenue_billions'].isna()].copy()
        removed_nan = before - len(rev_data)

        # (calendar_year, period) 중복 제거
        before2 = len(rev_data)
        rev_data = rev_data.drop_duplicates(subset=['calendar_year', 'period'], keep='first').reset_index(drop=True)
        removed_dup = before2 - len(rev_data)

        print(f"[clean_rev_data] removed rows → revenue NaN: {removed_nan}, duplicates: {removed_dup}")

        # ★ 정제 후 datetime 타입 재확인
        rev_data['date_month_end'] = pd.to_datetime(rev_data['date_month_end'], errors='coerce')


        # 기간 미달 체크
        if len(rev_data) < 44:
            print(f"[{ticker}] SKIP: 데이터 기간 미달 ({len(rev_data)}건 < 44건)")
            fail_count += 1
            continue

        print(f"[{ticker}] 매출 데이터 충족: {len(rev_data)}건")

        # ========================
        # 4. 매출 예측
        # ========================
        periods = 4

        # SARIMA
        sarima_df, results = sarima.run_sarima_prediction(
            rev_data,
            forecast_quarters=periods,
            exog_col=None
        )
        sarima_df = sarima_df.sort_values("date_month_end").set_index("date_month_end")

        # LSTM
        lstm_raw_df, lstm_results_4q = lstm_v2.run_lstm_revenue_prediction(
            rev_data, ticker=ticker, prediction_quarters=4
        )
        lstm_df = lstm_raw_df.drop_duplicates(subset=['revenue_billions_lstm_forecast'], keep='last')

        # Prophet
        prophet_raw_df, res_4q = prophet_v3.run_prophet_revenue_only(
            rev_data, ticker=ticker, prediction_quarters=4
        )

        # Exponential Smoothing
        es_raw_df, res_q4 = esmod.run_es_revenue_quarterly(
            rev_data, ticker=ticker, prediction_quarters=4
        )

         # ========================
        # 5. FMP 시가총액 데이터 수집
        # ========================
        print(f"[{ticker}] 5. FMP 시가총액 데이터 수집 중...")

        all_market_data = []
        current_year = pd.Timestamp.now().year

        for year in range(2010, current_year + 1):
            start_date_str = f"{year}-01-01"
            end_date_str = f"{year}-12-31"
            url = f"https://financialmodelingprep.com/api/v3/historical-market-capitalization/{ticker}"
            params = {'from': start_date_str, 'to': end_date_str, 'apikey': api_key}

            try:
                response = requests.get(url, params=params, timeout=30)
                if response.status_code == 200:
                    data = response.json()
                    if data and isinstance(data, list):
                        all_market_data.extend(data)
                time.sleep(0.3)
            except Exception:
                continue

        if not all_market_data:
            print(f"[{ticker}] ERROR: FMP 시가총액 데이터 수집 실패")
            fail_count += 1
            continue

        # 일별 데이터를 월별로 변환
        market_df = pd.DataFrame(all_market_data)
        market_df['date'] = pd.to_datetime(market_df['date'])
        market_df = market_df.sort_values('date')
        market_df['year_month'] = market_df['date'].dt.to_period('M')

        monthly_market_data = []
        for year_month in market_df['year_month'].unique():
            month_data = market_df[market_df['year_month'] == year_month]
            last_day_data = month_data.loc[month_data['date'].idxmax()]
            monthly_market_data.append({
                'ticker': ticker,
                'date': last_day_data['date'],
                'market_cap': last_day_data['marketCap'],
                'market_cap_billions': round(last_day_data['marketCap'] / 1_000_000_000, 2),
            })

        fmp_market_df = pd.DataFrame(monthly_market_data)

           # ★ datetime 타입으로 명시적 변환
        # fmp_revenue_df['date'] = pd.to_datetime(fmp_revenue_df['date'])
        # fmp_revenue_df['date_month_end'] = fmp_revenue_df['date'].apply(convert_to_month_end)

        fmp_market_df['date_month_end'] = convert_date_to_month_end(fmp_market_df['date'])

        # ★ datetime 타입 보장
        fmp_market_df['date_month_end'] = pd.to_datetime(fmp_market_df['date_month_end'])

        fmp_market_df = (fmp_market_df
                         .drop_duplicates(subset=['date_month_end'])
                         .sort_values('date_month_end')
                         .reset_index(drop=True))

        print(f"[{ticker}] FMP 시가총액 데이터: {len(fmp_market_df)}건")

        # ========================
        # 6. DB 시가총액 병합
        # ========================
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )

        query = f"""
        SELECT date, ticker, me
        FROM US_fundm
        WHERE ticker = '{ticker}'
        AND me IS NOT NULL
        AND date <= '2024-12-31'
        ORDER BY date ASC
        """

        db_market_df = pd.read_sql(query, con=engine)
        engine.dispose()

        # db_market_df = _safe_get_db_market_df(ticker, db_info)

        if db_market_df.empty:
            print(f"[{ticker}] INFO: DB 시가총액 데이터 없음 → FMP 데이터만 사용")
            merged_market_df = fmp_market_df.copy()
            merged_market_df['market_cap_billions_from_db'] = np.nan
        else:
            db_market_df['date'] = pd.to_datetime(db_market_df['date'])
            db_market_df['market_cap_billions'] = db_market_df['me'] / 1000

            # ★ DB 시가총액도 동일한 방식으로 변환
            db_market_df['date_month_end'] = convert_date_to_month_end(db_market_df['date'])
            db_market_df = db_market_df.drop_duplicates(subset=['date_month_end'], keep='first')

            # ★ datetime 타입 보장
            db_market_df['date_month_end'] = pd.to_datetime(db_market_df['date_month_end'])

            db_market_df = db_market_df[['ticker', 'date', 'date_month_end', 'market_cap_billions']]

            db_market_df_renamed = db_market_df.rename(
                columns={'market_cap_billions': 'market_cap_billions_from_db'}
            )

            merged_market_df = fmp_market_df.merge(
                db_market_df_renamed[['date_month_end', 'market_cap_billions_from_db']],
                on='date_month_end',
                how='left'
            )

        # ========================
        # 7. TTM 및 PSR 계산
        # ========================
        # ★ 병합 전 양쪽 모두 datetime 타입 보장
        merged_market_df['date_month_end'] = pd.to_datetime(merged_market_df['date_month_end'], errors='coerce')
        rev_data['date_month_end'] = pd.to_datetime(rev_data['date_month_end'], errors='coerce')

        enhanced_merged_df = pd.merge(
            merged_market_df[['date_month_end', 'market_cap_billions']],
            rev_data,
            on='date_month_end',
            how='outer'
        )

        market_cap_resize = enhanced_merged_df[['date_month_end', 'market_cap_billions', 'ticker', 'revenue_billions']].copy()
        market_cap_resize.dropna(subset=['market_cap_billions'], inplace=True)
        market_cap_resize.ffill(limit=2, inplace=True)
        market_cap_resize = market_cap_resize[
            (market_cap_resize['date_month_end'] >= start_date_month) &
            (market_cap_resize['date_month_end'] <= end_date_month)
        ]
        market_cap_resize = market_cap_resize.dropna(axis=0)

        # TTM 계산
        market_cap_resize['date_month_end'] = pd.to_datetime(market_cap_resize['date_month_end'], errors='coerce')
        market_cap_resize = market_cap_resize.sort_values(['date_month_end']).reset_index(drop=True)
        market_cap_resize = market_cap_resize.sort_values(['ticker', 'date_month_end']).reset_index(drop=True)

        market_cap_resize['revenue_ttm'] = (
            market_cap_resize.groupby('ticker')['revenue_billions']
            .rolling(window=4, min_periods=1)
            .sum()
            .reset_index(0, drop=True)
        )

        market_cap_resize['revenue_ttm_billions'] = market_cap_resize['revenue_ttm']
        market_cap_resize['revenue_ttm_shift'] = market_cap_resize.groupby('ticker')['revenue_ttm_billions'].shift(2)
        market_cap_resize['PSR_ttm'] = market_cap_resize['market_cap_billions'] / market_cap_resize['revenue_ttm_shift']
        market_cap_resize['PSR_ttm'] = market_cap_resize['PSR_ttm'].replace([np.inf, -np.inf], np.nan)

        enhanced_merged_df_with_ttm = market_cap_resize.copy()

        # ========================
        # 8. PSR 예측
        # ========================
        # SARIMA PSR
        psr_sarima_df, psr_12_res = sarima.run_sarima_psr_only(
            df=enhanced_merged_df_with_ttm,
            periods=12,
            target_col="PSR_ttm",
            analysis_start="2012-06-01",
            warmup_months=6,
            fill_method="interpolate",
            ic="aic"
        )

        # LSTM PSR
        psr_lstm_df, psr_results = lstm_v2.run_lstm_psr_prediction(
            enhanced_merged_df_with_ttm, ticker=ticker, prediction_months=12
        )

        # Prophet PSR
        psr_prophet_df, psr_res = prophet_v3.run_prophet_psr_only(
            enhanced_merged_df_with_ttm, ticker=ticker, prediction_months=12
        )

        # ES PSR
        psr_es_df, psr_res_es = esmod.run_es_psr_only(
            df=enhanced_merged_df_with_ttm,
            ticker=ticker,
            prediction_months=12,
            start_date=None
        )

        # ========================
        # 9. Valuation 종합
        # ========================
        sarima_resize_df = sarima_df[['ticker', 'revenue_billions_sarima_noexog']].copy()
        lstm_resize_df = lstm_df[['revenue_billions_lstm_forecast']].copy()
        prophet_resize_df = prophet_raw_df[['revenue_billions_prophet_forecast']].copy()
        es_resize_df = es_raw_df[['revenue_billions_esq_forecast']].copy()

        revenue_forecast_df = pd.concat([sarima_resize_df, lstm_resize_df, prophet_resize_df, es_resize_df], axis=1)

        psr_sarima_resiae = psr_sarima_df[['PSR_ttm_sarima_forecast']]
        psr_lstm_resiae = psr_lstm_df[['PSR_ttm_lstm_forecast']]
        psr_prophet_resiae = psr_prophet_df[['PSR_prophet_forecast_noexog']]
        psr_es_resiae = psr_es_df[['PSR_es_forecast']]

        psr_forecast_df = pd.concat([psr_sarima_resiae, psr_lstm_resiae, psr_prophet_resiae, psr_es_resiae], axis=1)

        # Revenue TTM 계산
        d = revenue_forecast_df.copy()
        if 'date_month_end' not in d.columns:
            d = d.reset_index().rename(columns={'index': 'date_month_end'})
        d['date_month_end'] = pd.to_datetime(d['date_month_end'])

        d['ticker'] = d['ticker'].ffill().bfill()
        d = d.sort_values(['ticker', 'date_month_end']).reset_index(drop=True)

        rev_cols = [c for c in d.columns if 'revenue_billions' in c]
        row_mean = d[rev_cols].mean(axis=1, skipna=True)
        for c in rev_cols:
            d[c] = d[c].fillna(row_mean)

        for c in rev_cols:
            ttm_col = f"{c}_ttm"
            d[ttm_col] = (
                d.groupby('ticker', group_keys=False)[c]
                 .rolling(window=4, min_periods=1)
                 .sum()
                 .reset_index(level=0, drop=True)
            )

        d = d.set_index('date_month_end')
        revenue_forecast_ = d
        revenue_forecast_ttm = revenue_forecast_.filter(like='_ttm')
        revenue_forecast_ttm['ticker'] = ticker

        valuation_df = pd.concat([revenue_forecast_ttm, psr_forecast_df], axis=1)

        # ========================
        # 10. Valuation 계산
        # ========================
        valuation_filled = valuation_df.copy()

        cols_to_fill = ['ticker'] + [c for c in valuation_filled.columns if 'revenue' in c]
        valuation_filled[cols_to_fill] = valuation_filled[cols_to_fill].ffill(limit=2)

        valuation_filled['sarima_valuation'] = (
            valuation_filled['revenue_billions_sarima_noexog_ttm'] *
            valuation_filled['PSR_ttm_sarima_forecast']
        )

        valuation_filled['lstm_valuation'] = (
            valuation_filled['revenue_billions_lstm_forecast_ttm'] *
            valuation_filled['PSR_ttm_lstm_forecast']
        )

        valuation_filled['prophet_valuation'] = (
            valuation_filled['revenue_billions_prophet_forecast_ttm'] *
            valuation_filled['PSR_prophet_forecast_noexog']
        )

        valuation_filled['es_valuation'] = (
            valuation_filled['revenue_billions_esq_forecast_ttm'] *
            valuation_filled['PSR_es_forecast']
        )

        # ========================
        # 11. 마지막 15개월 추출
        # ========================
        if 'date_month_end' in valuation_filled.columns:
            valuation_filled = valuation_filled.sort_values('date_month_end')
            valuation_result = valuation_filled.groupby('ticker').tail(15).reset_index(drop=True)
        else:
            valuation_filled = valuation_filled.sort_index()
            valuation_result = valuation_filled.groupby('ticker').tail(15).reset_index()

        # 측정 날짜 추가
        valuation_result['measurement_date'] = measurement_date
        valuation_result['ticker'] = ticker
        print(f"[{ticker}] SUCCESS: Valuation 계산 완료 ({len(valuation_result)}건)")

        all_results.append(valuation_result)
        success_count += 1
        print(f"[{ticker}] ✓ 성공")

    except Exception as e:
        print(f"[{ticker}] ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        fail_count += 1
        print(f"[{ticker}] ✗ 실패")

# ========================
# 결과 종합
# ========================
print(f"\n{'='*80}")
print(f"처리 완료:")
print(f"  성공: {success_count}개")
print(f"  실패: {fail_count}개")
print(f"{'='*80}\n")

if not all_results:
    print("WARNING: 성공한 결과가 없습니다.")
else:
    # 모든 결과 결합
    final_results = pd.concat(all_results, ignore_index=True)

    print("\n최종 결과:")
    print(final_results.head(20))
    print(f"\n총 {len(final_results)}개 행")

    # CSV 저장
    output_file = f"valuation_results_{pd.Timestamp.today().strftime('%Y%m%d')}.csv"
    final_results.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\n결과 저장: {output_file}")

Using project path: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy

Multi-Ticker Valuation 시작
총 1개 종목
기간: 2011-03-01 ~ 2025-09-30
측정일: 2025-10-07


[1/1] 처리 중: MU

처리 시작: MU

[MU] 1. FMP 매출 데이터 수집 중...

[MU] 중복 제거 전: 161건
[MU] 중복 제거 후 FMP 매출 데이터: 161건

[MU] 2. DB 매출 데이터 병합 중...
[MU] DB 원본 데이터: 554건
[MU] DB 연속 중복 제거 후: 107건
[MU] 병합 후 데이터: 252건
[MU] 기간 필터링 후 (2011-03-01 이후): 107건
[MU] 최종 데이터 건수 (정제 전): 107건
[clean_rev_data] removed rows → revenue NaN: 0, duplicates: 47
[MU] 매출 데이터 충족: 60건


10:58:23 - cmdstanpy - INFO - Chain [1] start processing
10:58:23 - cmdstanpy - INFO - Chain [1] done processing


[MU] 5. FMP 시가총액 데이터 수집 중...
[MU] FMP 시가총액 데이터: 190건
[MU] INFO: DB 시가총액 데이터 없음 → FMP 데이터만 사용


10:59:10 - cmdstanpy - INFO - Chain [1] start processing


[INFO] 예측 시작일: 2024-05-31 | 데이터 마지막 월: 2024-04-30


10:59:10 - cmdstanpy - INFO - Chain [1] done processing


[MU] SUCCESS: Valuation 계산 완료 (15건)
[MU] ✓ 성공

처리 완료:
  성공: 1개
  실패: 0개


최종 결과:
        index  revenue_billions_sarima_noexog_ttm  \
0  2024-08-31                           25.110000   
1  2024-09-30                           25.110000   
2  2024-10-31                           25.110000   
3  2024-11-30                           29.090000   
4  2024-12-31                           29.090000   
5  2025-01-31                           29.090000   
6  2025-02-28                           31.320000   
7  2025-03-31                           31.320000   
8  2025-04-30                           31.320000   
9  2025-05-31                           33.810000   
10 2025-08-31                           37.370000   
11 2025-11-30                           41.200702   
12 2026-02-28                           45.484262   
13 2026-05-31                           48.950370   
14 2026-08-31                           50.710028   

    revenue_billions_lstm_forecast_ttm  revenue_billions_prophet_forec

In [23]:
rev_data

,ticker,date_x,calendar_year,period,revenue,revenue_billions,date_month_end,date_y,revenue_billions_y
0,MU,2011-03-03,2011,Q2,2.257000e+09,2.260,2011-03-03,NaT,NaN
1,MU,NaT,NaN,NaN,NaN,2.139,2011-05-31,2011-05-31,2.139
2,MU,2011-06-02,2011,Q3,2.139000e+09,2.140,2011-06-02,NaT,NaN
3,MU,2011-09-01,2011,Q4,2.140000e+09,2.140,2011-09-01,NaT,NaN
4,MU,2011-12-01,2012,Q1,2.090000e+09,2.090,2011-12-01,NaT,NaN
5,MU,2012-03-01,2012,Q2,2.067000e+09,2.070,2012-03-01,NaT,NaN
6,MU,2012-05-31,2012,Q3,2.172000e+09,2.170,2012-05-31,2012-05-31,2.172
7,MU,2012-08-30,2012,Q4,1.963000e+09,1.960,2012-08-30,NaT,NaN
8,MU,2012-11-29,2013,Q1,1.834000e+09,1.830,2012-11-29,NaT,NaN
9,MU,2013-02-28,2013,Q2,2.078000e+09,2.080,2013-02-28,2013-02-28,2.078


In [12]:
rev_data

,ticker,date_x,calendar_year,period,revenue,revenue_billions,date_month_end,date_y,revenue_billions_y
140,AMD,NaT,NaN,NaN,NaN,1.613,2011-03-31,2011-03-31,1.613
141,AMD,2011-04-02,2011,Q1,1.613000e+09,1.610,2011-04-02,NaT,NaN
142,AMD,NaT,NaN,NaN,NaN,1.574,2011-06-30,2011-06-30,1.574
143,AMD,2011-07-02,2011,Q2,1.574000e+09,1.570,2011-07-02,NaT,NaN
144,AMD,NaT,NaN,NaN,NaN,1.690,2011-09-30,2011-09-30,1.690
...,...,...,...,...,...,...,...,...,...
241,AMD,NaT,NaN,NaN,NaN,7.658,2024-12-31,2024-12-31,7.658
242,AMD,2025-03-29,2025,Q1,7.438000e+09,7.440,2025-03-29,NaT,NaN
243,AMD,NaT,NaN,NaN,NaN,7.438,2025-03-31,2025-03-31,7.438
244,AMD,2025-06-28,2025,Q2,7.685000e+09,7.680,2025-06-28,NaT,NaN


In [24]:
fmp_revenue_df

,ticker,date,calendar_year,period,revenue,revenue_billions,date_month_end
0,MU,1985-08-31,1985,Q4,6200000,0.01,1985-08-31
1,MU,1985-11-30,1986,Q1,5000000,0.01,1985-11-30
2,MU,1986-02-28,1986,Q2,9400000,0.01,1986-02-28
3,MU,1986-05-31,1986,Q3,14400000,0.01,1986-05-31
4,MU,1986-08-31,1986,Q4,20000000,0.02,1986-08-31
...,...,...,...,...,...,...,...
156,MU,2024-08-29,2024,Q4,7750000000,7.75,2024-08-29
157,MU,2024-11-28,2025,Q1,8709000000,8.71,2024-11-28
158,MU,2025-02-27,2025,Q2,8053000000,8.05,2025-02-27
159,MU,2025-05-29,2025,Q3,9301000000,9.30,2025-05-29
